# Re-evaluate Baseline theo Code-Switching Subset - ViGoEmotions

Notebook này dùng để chạy trên Kaggle. Mục tiêu: đọc ViGoEmotions từ Kaggle Dataset trong `/kaggle/input`, gắn nhóm code-switching (trộn lẫn ngôn ngữ) bằng rule-based detector, chạy inference bằng baseline đã train, rồi báo cáo metric theo từng subset: `pure_vi`, `code_switched`, `english_mixed`, `chinese_mixed`, `multi_mixed`, `other_foreign`.

Định nghĩa code-switching ở đây khớp với `vigo_code_switching_analysis.ipynb`: chỉ tính việc trộn ngôn ngữ khác (Anh/Trung/khác) vào tiếng Việt, không tính emoji/teencode.

In [ ]:
!pip -q install datasets transformers accelerate evaluate scikit-learn

In [ ]:
import os
import re
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, hamming_loss
from tqdm.auto import tqdm

MODEL_NAME_OR_PATH = "/kaggle/input/YOUR_BASELINE_MODEL_PATH"
# Ví dụ:
# MODEL_NAME_OR_PATH = "/kaggle/input/xlm-r-vigoemotions/best_model"
# MODEL_NAME_OR_PATH = "/kaggle/working/vigo_baseline_outputs/xlm-r-vigoemotions/best_model"

# Sửa path này theo Kaggle Dataset bạn add vào notebook.
DATA_DIR = "/kaggle/input/YOUR_VIGOEMOTIONS_DATASET"

MAX_LENGTH = 128
BATCH_SIZE = 32
THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LABEL_NAMES = [
    "amusement", "excitement", "joy", "love", "desire", "optimism", "caring",
    "pride", "admiration", "gratitude", "relief", "approval", "realization",
    "surprise", "curiosity", "confusion", "fear", "nervousness", "remorse",
    "embarrassment", "disappointment", "sadness", "grief", "disgust", "anger",
    "annoyance", "disapproval", "neutral"
]

NUM_LABELS = len(LABEL_NAMES)


## Load ViGoEmotions từ Kaggle Dataset

Trên Kaggle, add dataset ViGoEmotions vào notebook, rồi sửa `DATA_DIR` cho đúng folder trong `/kaggle/input/...`. Notebook hỗ trợ file riêng `train/val/test` hoặc một file chung có cột `split`/`set`.

In [ ]:
def normalize_split_name(split_name):
    if split_name in ["validation", "dev"]:
        return "val"
    return split_name

def read_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        xls = pd.ExcelFile(path)
        if {"train", "val", "test"}.issubset(set(xls.sheet_names)):
            frames = []
            for sheet in ["train", "val", "test"]:
                df = pd.read_excel(path, sheet_name=sheet)
                df["split"] = sheet
                frames.append(df)
            return pd.concat(frames, ignore_index=True)
        return pd.read_excel(path, sheet_name=xls.sheet_names[0])
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in [".json", ".jsonl"]:
        return pd.read_json(path, lines=(suffix == ".jsonl"))
    raise ValueError(f"Unsupported file type: {path}")

def find_data_files(data_dir):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            f"DATA_DIR does not exist: {data_dir}\n"
            "Set DATA_DIR to your Kaggle dataset folder, e.g. /kaggle/input/vigoemotions"
        )
    exts = {".csv", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"}
    files = [p for p in data_dir.rglob("*") if p.is_file() and p.suffix.lower() in exts]
    if not files:
        raise FileNotFoundError(f"No supported data files found under: {data_dir}")
    return files

def standardize_columns(df):
    df = df.copy()
    lower_map = {c.lower(): c for c in df.columns}

    if "text" not in df.columns:
        for candidate in ["comment", "sentence", "content", "clean_text"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "text"})
                break

    if "labels" not in df.columns:
        for candidate in ["label", "emotion", "emotions", "target", "targets"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "labels"})
                break

    if "split" not in df.columns:
        for candidate in ["set", "data_split"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "split"})
                break

    missing = [c for c in ["text", "labels"] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}. Available columns: {list(df.columns)}")

    return df

files = find_data_files(DATA_DIR)
print("Found files:")
for f in files:
    print("-", f)

named_split_frames = []
single_frames = []

for file_path in files:
    df = standardize_columns(read_table(file_path))
    name = file_path.stem.lower()

    if "split" in df.columns:
        single_frames.append(df)
    elif "train" in name:
        df["split"] = "train"
        named_split_frames.append(df)
    elif any(k in name for k in ["val", "dev", "validation"]):
        df["split"] = "val"
        named_split_frames.append(df)
    elif "test" in name:
        df["split"] = "test"
        named_split_frames.append(df)

if single_frames:
    data = pd.concat(single_frames, ignore_index=True)
elif named_split_frames:
    data = pd.concat(named_split_frames, ignore_index=True)
else:
    raise ValueError(
        "Cannot infer splits. Use files named train/val/test, "
        "or provide a single file with column split/set."
    )

data = standardize_columns(data)
data["split"] = data["split"].astype(str).str.lower().map(normalize_split_name)

print(data.shape)
print(data["split"].value_counts())
data.head()


In [ ]:
required_splits = {"train", "val", "test"}
found_splits = set(data["split"].dropna().unique())
missing_splits = required_splits - found_splits
if missing_splits:
    raise ValueError(f"Missing splits: {missing_splits}. Found splits: {found_splits}")

data.head()


In [ ]:
def parse_labels(x, num_labels=NUM_LABELS):\n    if isinstance(x, np.ndarray):\n        x = x.tolist()\n\n    if isinstance(x, str):\n        try:\n            x = ast.literal_eval(x)\n        except Exception:\n            x = [int(i) for i in re.findall(r"\\d+", x)]\n\n    if isinstance(x, (list, tuple)):\n        x = list(x)\n\n        if len(x) == num_labels and set(np.unique(x)).issubset({0, 1, 0.0, 1.0}):\n            return np.array(x, dtype=int)\n\n        y = np.zeros(num_labels, dtype=int)\n        for idx in x:\n            if 0 <= int(idx) < num_labels:\n                y[int(idx)] = 1\n        return y\n\n    y = np.zeros(num_labels, dtype=int)\n    try:\n        y[int(x)] = 1\n    except Exception:\n        pass\n    return y\n\ndata["y_true"] = data["labels"].apply(parse_labels)

## Rule-Based Code-Switching Detector

In [ ]:
import re
import unicodedata

# ===========================================================================
# Rule-based CROSS-LANGUAGE code-switching detector
# ---------------------------------------------------------------------------
# Ở đây code-switching CHỈ gồm việc trộn ngôn ngữ khác vào tiếng Việt
# (chủ yếu Anh - Việt - Trung). Emoji, emoticon và teencode/tiếng lóng thuần
# Việt KHÔNG được tính là code-switching (teencode chỉ dùng để loại trừ, tránh
# nhận nhầm là tiếng Anh). Từ nước ngoài đã được "Việt hoá" / phiên âm (ví dụ
# tiếng Trung 谢谢 -> "xia xìa", 你好 -> "nỉ hảo") vẫn được tính là
# code-switching thông qua ZH_TRANSLIT_SEQS bên dưới.
# ===========================================================================

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'(?<!\w)@[\w_]+')
HASHTAG_RE = re.compile(r'(?<!\w)#[\w_]+')
# Token chữ Latin (gồm cả dấu tiếng Việt)
WORD_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+(?:[-'][A-Za-zÀ-ỹĐđ]+)?", re.UNICODE)
ASCII_ALPHA_RE = re.compile(r'^[A-Za-z]+$')
VIET_CHARS_RE = re.compile(r'[ăâđêôơưáàảãạấầẩẫậắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]', re.IGNORECASE)
EMOJI_RE = re.compile('[\U0001F300-\U0001FAFF\U00002700-\U000027BF\U00002600-\U000026FF]+')

# Hệ chữ nước ngoài không nhập nhằng
HAN_RE = re.compile('[㐀-䶿一-鿿豈-﫿]+')   # chữ Hán (Trung/Nhật kanji)
HANGUL_RE = re.compile('[가-힣]+')                          # Hangul (Hàn)
KANA_RE = re.compile('[぀-ヿ]+')                            # Kana (Nhật)

# Tiếng Việt (đã bỏ dấu) + teencode thuần Việt -> KHÔNG coi là tiếng nước ngoài.
# Chỉ dùng để loại trừ, giúp is_english_like không bắt nhầm teencode thành tiếng Anh.
VI_EXCLUDE = set('''
toi tui tao may ban minh chung nguoi la thi ma va hay nhung vi voi cho cua trong tren duoi nay kia do day roi chua
khong ko k k0 kh khum hong hok hem duoc dc dk dx di den ve an uong lam hoc choi xem nghe noi nghi biet thay ghe qua
troi rat hoi luon nua nha nhe dang da se mot hai ba bon nam sau bay tam chin muoi
j z dz zj zi v vs vl vc vcl vkl vãi vai cx cg cung chu chz dm dcm cmt cmm clm mik mk mjk mn mng ae ny iu yeu lun
rui ruoi bt bth bthuong thik ib rep kb tks pls plz wa qa qá bn b t m r ui z0 hj hjhj kk kkk haha hihi hehe huhu
'''.split())

# Từ tiếng Anh phổ biến (gồm cả loanword giới trẻ hay chèn vào câu tiếng Việt).
EN_COMMON = set('''
love like hate happy sad angry fear sorry thanks thank please ok okay yes no good bad best worst cute nice cool hot
trend trending fan idol team movie music video clip live stream comment share post status story drama fake real toxic
cringe brainwash anti respect support follow unfollow block game gaming win lose loser boy girl baby bro sis friend
family crush deadline job work school class online offline ship shipper couple hello bye view sub subscribe show style
admin facebook youtube tiktok perfect legend wow king queen
'''.split())

EN_SUFFIX_RE = re.compile(r'(ing|tion|ment|ness|ous|ive|er|ed|ly|ship|ful)$')

# Phiên âm tiếng Trung hay gặp trong bình luận tiếng Việt (đã bỏ dấu; cho phép
# viết liền hoặc cách khoảng). Thêm cụm mới vào đây khi cần mở rộng.
ZH_TRANSLIT_SEQS = [
    ['xie', 'xie'], ['xia', 'xia'],            # 谢谢  cảm ơn
    ['ni', 'hao'], ['nin', 'hao'],             # 你好  xin chào
    ['da', 'jia', 'hao'],                      # 大家好 chào mọi người
    ['wo', 'ai', 'ni'],                        # 我爱你 anh yêu em
    ['zai', 'jian'],                           # 再见  tạm biệt
    ['dui', 'bu', 'qi'],                       # 对不起 xin lỗi
    ['mei', 'guan', 'xi'],                     # 没关系 không sao
    ['jia', 'you'],                            # 加油  cố lên
    ['bao', 'bei'],                            # 宝贝  cục cưng
    ['gan', 'bei'],                            # 干杯  cạn ly
    ['piao', 'liang'],                         # 漂亮  xinh đẹp
    ['shuai', 'ge'],                           # 帅哥  anh đẹp trai
    ['mei', 'nu'],                             # 美女  gái xinh
    ['shen', 'me'],                            # 什么  cái gì
    ['wei', 'shen', 'me'],                     # 为什么 tại sao
    ['zhen', 'de'],                            # 真的  thật sự
    ['gong', 'xi', 'fa', 'cai'],               # 恭喜发财 chúc mừng năm mới
    ['wan', 'an'],                             # 晚安  ngủ ngon
    ['zao', 'an'],                             # 早安  chào buổi sáng
    ['wo', 'men'], ['ni', 'men'],              # 我们/你们 chúng ta/các bạn
    ['tai', 'hao', 'le'],                      # 太好了 tuyệt quá
]
ZH_TRANSLIT_PATTERNS = [re.compile(r'\b' + r'\s*'.join(seq) + r'\b') for seq in ZH_TRANSLIT_SEQS]
ZH_TRANSLIT_LABELS = [' '.join(seq) for seq in ZH_TRANSLIT_SEQS]


def fold_accents(text):
    text = unicodedata.normalize('NFD', str(text))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('Đ', 'D').replace('đ', 'd').lower()


def is_english_like(token):
    t = token.lower()
    if not ASCII_ALPHA_RE.match(token):
        return False
    if len(t) <= 2:
        return False
    if t in EN_COMMON:
        return True
    if t in VI_EXCLUDE:
        return False
    return bool(EN_SUFFIX_RE.search(t))


def detect_chinese(raw, folded):
    han_tokens = HAN_RE.findall(raw)
    translit = [lab for pat, lab in zip(ZH_TRANSLIT_PATTERNS, ZH_TRANSLIT_LABELS) if pat.search(folded)]
    return han_tokens + translit


def analyze_text(text):
    raw = str(text)
    folded = fold_accents(raw)
    word_tokens = WORD_RE.findall(raw)
    num_word_tokens = len(word_tokens)

    english_tokens = [tok for tok in word_tokens if is_english_like(tok)]
    chinese_tokens = detect_chinese(raw, folded)
    other_foreign_tokens = HANGUL_RE.findall(raw) + KANA_RE.findall(raw)

    has_english = len(english_tokens) > 0
    has_chinese = len(chinese_tokens) > 0
    has_other_foreign = len(other_foreign_tokens) > 0

    langs = []
    if has_english:
        langs.append('en')
    if has_chinese:
        langs.append('zh')
    if has_other_foreign:
        langs.append('other')

    if not langs:
        cs_group = 'pure_vi'
    elif len(langs) >= 2:
        cs_group = 'multi_mixed'
    elif langs[0] == 'en':
        cs_group = 'english_mixed'
    elif langs[0] == 'zh':
        cs_group = 'chinese_mixed'
    else:
        cs_group = 'other_foreign'

    return pd.Series({
        'num_tokens': num_word_tokens,
        'has_vietnamese_diacritic': bool(VIET_CHARS_RE.search(raw)),
        'has_english': has_english,
        'english_count': len(english_tokens),
        'english_ratio': len(english_tokens) / max(num_word_tokens, 1),
        'english_tokens': ', '.join(english_tokens[:20]),
        'has_chinese': has_chinese,
        'chinese_count': len(chinese_tokens),
        'chinese_tokens': ', '.join(chinese_tokens[:20]),
        'has_other_foreign': has_other_foreign,
        'other_foreign_tokens': ', '.join(other_foreign_tokens[:20]),
        'foreign_langs': ', '.join(langs),
        'emoji_count': len(EMOJI_RE.findall(raw)),
        'url_count': len(URL_RE.findall(raw)),
        'mention_count': len(MENTION_RE.findall(raw)),
        'hashtag_count': len(HASHTAG_RE.findall(raw)),
        'is_code_switched': len(langs) > 0,
        'cs_group': cs_group,
    })


cs_features = data["text"].apply(analyze_text)
data = pd.concat([data, cs_features], axis=1)

display(data["cs_group"].value_counts())
display(pd.crosstab(data["split"], data["cs_group"], margins=True))


## Load Baseline Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, use_fast=False)\n\nmodel = AutoModelForSequenceClassification.from_pretrained(\n    MODEL_NAME_OR_PATH,\n    num_labels=NUM_LABELS,\n    problem_type="multi_label_classification",\n    ignore_mismatched_sizes=True\n)\n\nmodel.to(DEVICE)\nmodel.eval()

## Inference trên Test Set

In [ ]:
@torch.no_grad()\ndef predict_proba(texts):\n    all_probs = []\n\n    for i in tqdm(range(0, len(texts), BATCH_SIZE)):\n        batch_texts = texts[i:i + BATCH_SIZE]\n        enc = tokenizer(\n            batch_texts,\n            padding=True,\n            truncation=True,\n            max_length=MAX_LENGTH,\n            return_tensors="pt"\n        )\n        enc = {k: v.to(DEVICE) for k, v in enc.items()}\n        outputs = model(**enc)\n        probs = torch.sigmoid(outputs.logits).detach().cpu().numpy()\n        all_probs.append(probs)\n\n    return np.vstack(all_probs)\n\ntest_df = data[data["split"] == "test"].reset_index(drop=True)\ny_true = np.vstack(test_df["y_true"].values)\ny_prob = predict_proba(test_df["text"].astype(str).tolist())\ny_pred = (y_prob >= THRESHOLD).astype(int)\n\nprint(y_true.shape, y_pred.shape)

## Evaluate theo Subset

In [ ]:
def multilabel_metrics(y_true, y_pred):
    return {
        "n_samples": len(y_true),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }

test_eval = test_df.copy()
test_eval["row_id"] = np.arange(len(test_eval))

subsets = {
    "all": np.ones(len(test_eval), dtype=bool),
    "pure_vi": test_eval["cs_group"].eq("pure_vi").values,
    "code_switched": test_eval["is_code_switched"].eq(True).values,
    "english_mixed": test_eval["cs_group"].eq("english_mixed").values,
    "chinese_mixed": test_eval["cs_group"].eq("chinese_mixed").values,
    "multi_mixed": test_eval["cs_group"].eq("multi_mixed").values,
    "other_foreign": test_eval["cs_group"].eq("other_foreign").values,
}

subset_rows = []

for subset_name, mask in subsets.items():
    if mask.sum() == 0:
        continue
    metrics = multilabel_metrics(y_true[mask], y_pred[mask])
    metrics["subset"] = subset_name
    metrics["ratio"] = mask.mean()
    subset_rows.append(metrics)

subset_metrics_df = pd.DataFrame(subset_rows)
subset_metrics_df = subset_metrics_df[[
    "subset", "n_samples", "ratio",
    "micro_f1", "macro_f1", "weighted_f1",
    "micro_precision", "macro_precision",
    "micro_recall", "macro_recall",
    "hamming_loss", "subset_accuracy"
]]

display(subset_metrics_df)

In [ ]:
per_label_rows = []\n\nfor subset_name, mask in subsets.items():\n    if mask.sum() == 0:\n        continue\n\n    y_t = y_true[mask]\n    y_p = y_pred[mask]\n\n    f1_each = f1_score(y_t, y_p, average=None, zero_division=0)\n    precision_each = precision_score(y_t, y_p, average=None, zero_division=0)\n    recall_each = recall_score(y_t, y_p, average=None, zero_division=0)\n\n    for i, label in enumerate(LABEL_NAMES):\n        per_label_rows.append({\n            "subset": subset_name,\n            "label": label,\n            "support": int(y_t[:, i].sum()),\n            "precision": precision_each[i],\n            "recall": recall_each[i],\n            "f1": f1_each[i],\n        })\n\nper_label_metrics_df = pd.DataFrame(per_label_rows)\ndisplay(per_label_metrics_df.head())

In [ ]:
compare_df = subset_metrics_df[
    subset_metrics_df["subset"].isin(["pure_vi", "code_switched", "english_mixed", "chinese_mixed"])
].copy()

display(compare_df[["subset", "n_samples", "micro_f1", "macro_f1", "hamming_loss"]])

## Save Outputs

In [ ]:
OUT_DIR = "/kaggle/working/baseline_subset_eval"\nos.makedirs(OUT_DIR, exist_ok=True)\n\nsubset_metrics_df.to_csv(f"{OUT_DIR}/subset_metrics.csv", index=False)\nper_label_metrics_df.to_csv(f"{OUT_DIR}/per_label_metrics_by_subset.csv", index=False)\n\ntest_eval_out = test_eval.drop(columns=["y_true"]).copy()\n\nfor i, label in enumerate(LABEL_NAMES):\n    test_eval_out[f"true_{label}"] = y_true[:, i]\n    test_eval_out[f"prob_{label}"] = y_prob[:, i]\n    test_eval_out[f"pred_{label}"] = y_pred[:, i]\n\ntest_eval_out.to_csv(f"{OUT_DIR}/test_predictions_with_code_switching.csv", index=False)\n\nprint("Saved to:", OUT_DIR)\nprint(os.listdir(OUT_DIR))

In [ ]:
display(subset_metrics_df)\n\ndisplay(\n    per_label_metrics_df\n    .sort_values(["subset", "f1"], ascending=[True, True])\n    .groupby("subset")\n    .head(10)\n)